![NVIDIA Header](../../shared/assets/nvidia_header.png)

# NemoClaw with OpenClaw

### One managed deployment, three control planes: lifecycle, runtime, and agent harness

⏱ Estimated time: 45 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Identify the live sandbox managed by NemoClaw and verify that OpenClaw runs inside it</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Explain why <code>nemoclaw</code>, <code>openshell</code>, and <code>openclaw</code> commands answer different questions</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Locate the OpenClaw state root, workspace, skills, and session surfaces without exposing secrets</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Read the active OpenShell policy for the managed sandbox</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Confirm the sandbox can reach the configured inference route without revealing the API key</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Explain why policy changes are runtime operations, not prompt instructions</p>

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2024-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

## Setup

This setup verifies OpenShell, finds the course sandbox, and confirms the OpenClaw gateway can answer a health probe.

**Platform:** Validated NemoClaw host (Ubuntu 24.04, DGX Spark, or Apple Silicon macOS with Colima)

**Prerequisites:** Run `./setup.sh` and complete **[Sandbox Policies](../02-openshell-runtime/02_sandbox_policies.ipynb)**.

In [ ]:
import shlex
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "shared").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from shared.utils.display import run_command
from shared.utils.platform import (
    get_dashboard_base_url,
    openclaw_workspace_path,
    resolve_course_sandbox_name,
)
from shared.utils.setup import setup

setup(required_keys=[], require_node=True, require_openshell=True)
course_sandbox_name = resolve_course_sandbox_name()
workspace_path = openclaw_workspace_path()
print(f"Course sandbox: {course_sandbox_name}")
print(f"OpenClaw workspace path inside sandbox: {workspace_path}")
print(f"Dashboard base URL on this host: {get_dashboard_base_url()}")

---

## How the Layers Fit

NemoClaw creates the managed sandbox. OpenShell enforces the policy. OpenClaw runs inside that boundary. NemoClaw owns the host CLI and blueprint orchestration; OpenShell owns sandboxing, policy, port forwarding, and inference routing; OpenClaw owns the agent harness.

This course uses [OpenClaw](https://docs.openclaw.ai/start/openclaw) as the harness: a general-purpose, always-on assistant. Four ideas are enough to follow the rest of the course:

| Concept | What it is | Where you meet it |
|---|---|---|
| [SOUL.md](https://docs.openclaw.ai/concepts/soul) | A Markdown file that defines an agent's personality | Module 4, when you give an agent a persona |
| [SKILL.md](https://docs.openclaw.ai/tools/creating-skills) | A Markdown file that defines a capability an agent can use | Module 4, when you deploy API-backed skills |
| [Workspace](https://docs.openclaw.ai/concepts/agent-workspace) | Per-agent files for identity, user context, persistent notes, and local skills | Section 4 below |
| [Gateway](https://docs.openclaw.ai/gateway/index) | The OpenClaw service that answers health probes and serves sessions | Sections 2 and 6 below |

NemoClaw also manages Hermes Agent and Deep Agents Code. They run behind the same OpenShell boundary but are outside the scope of this distilled course.

---

## 1. Compare The Three Control Planes

Use this pattern when debugging:

| Command family | Layer | Best question |
|---|---|---|
| `nemoclaw` | Blueprint and lifecycle | What did the managed deployment configure? |
| `openshell` | Runtime and sandbox | Is the sandbox healthy and what policy applies? |
| `openclaw` | Agent harness | Is the agent gateway/config/session state healthy? |


In [ ]:
run_command("sh -lc 'nemoclaw status || nemoclaw agent status'", timeout=60)
run_command("openshell sandbox list", timeout=30)

---

## 2. Verify OpenClaw Inside The Sandbox

The host may have a CLI installed for convenience, but the course agent runs
inside the OpenShell sandbox.


In [ ]:
run_command(
    f"openshell sandbox exec -n {course_sandbox_name} -- sh -lc 'command -v openclaw; openclaw --version; openclaw gateway health || true'",
    timeout=60,
)

---

## 3. Confirm The Current State Surface

Older training often referred to the legacy OpenClaw data-root path. The current
NemoClaw manifest declares `/sandbox/.openclaw` as the OpenClaw config root.


In [ ]:
state_probe = (
    "test -d /sandbox/.openclaw && echo OPENCLAW_STATE_ROOT_PRESENT; "
    "test -f /sandbox/.openclaw/openclaw.json && echo OPENCLAW_CONFIG_PRESENT; "
    "find /sandbox/.openclaw -maxdepth 1 -type d -printf '%f\\n' | sort | sed -n '1,80p'"
)
run_command(
    f"openshell sandbox exec -n {course_sandbox_name} -- sh -lc {shlex.quote(state_probe)}",
    timeout=60,
)

---

## 4. Inspect Workspace Files

OpenClaw uses workspace files for identity, user context, persistent notes,
and local skills. Do not confuse these files with the complete config and
credential state under `/sandbox/.openclaw`.


In [ ]:
workspace_probe = (
    f"echo WORKSPACE={workspace_path}; "
    f"if [ -d {workspace_path} ]; then "
    "echo WORKSPACE_PRESENT; "
    f"files=$(find {workspace_path} -maxdepth 2 -type f | sort | sed -n '1,80p'); "
    'if [ -n "$files" ]; then printf \'%s\\n\' "$files"; else echo WORKSPACE_EMPTY; fi; '
    "else echo WORKSPACE_NOT_FOUND; fi"
)
run_command(
    f"openshell sandbox exec -n {course_sandbox_name} -- sh -lc {shlex.quote(workspace_probe)}",
    timeout=60,
)

---

## 5. Inspect Skills Without Installing Anything

This is a read-only inspection. The goal is to confirm that the harness sees a
skills surface, not to mutate the training environment.


In [ ]:
run_command(
    f"openshell sandbox exec -n {course_sandbox_name} --timeout 20 -- sh -lc 'openclaw skills list | sed -n 1,60p'",
    timeout=30,
)

---

## 6. Inspect Gateway Health And Optional Session State

Gateway health should be stable. Session output is live state and may take
longer to answer on a busy sandbox; when it does not return quickly, the
notebook reports that condition instead of treating it as a gateway failure.


In [ ]:
session_probe = (
    "openclaw gateway health || true; "
    "if timeout 15 sh -lc 'openclaw sessions 2>/dev/null | head -40'; then "
    ":; else echo SESSIONS_LIST_UNAVAILABLE_WITHIN_TIMEOUT; fi"
)
run_command(
    f"openshell sandbox exec -n {course_sandbox_name} --timeout 30 -- sh -lc {shlex.quote(session_probe)}",
    timeout=40,
)

---

## 7. Read The Active Policy

This is the policy OpenShell is enforcing now on the managed sandbox. It has the same
four domains you wrote by hand in the previous notebook, expanded from the NemoClaw
policy tier chosen at onboarding. Changing it is a runtime operation performed by the
operator; nothing the agent is told in a prompt can alter it.


In [ ]:
run_command(f"openshell policy get {course_sandbox_name} --full", timeout=60)

---

## 8. Probe The Inference Route

The request uses OpenShell's secret-resolution form. The cell should show HTTP
status and a short model summary, not the secret value or a full catalog dump.


In [ ]:
inference_probe = (
    "set +e; "
    "body=$(mktemp); "
    "status=$(curl -sS -o \"$body\" -w '%{http_code}' https://inference.local/v1/models "
    "-H 'Authorization: Bearer openshell:resolve:env:NVIDIA_API_KEY'); "
    "rc=$?; "
    "echo CURL_EXIT=$rc; "
    "echo HTTP_STATUS=$status; "
    "python3 -c 'import json, sys; "
    "p=sys.argv[1]; "
    "data=json.load(open(p)); "
    'models=data.get("data", []); '
    'print(f"MODEL_COUNT={len(models)}"); '
    '[print("MODEL_ID=" + str(m.get("id", ""))) for m in models[:20]]\' "$body" '
    '|| head -c 2000 "$body"; '
    'rm -f "$body"; '
    'if [ "$rc" -ne 0 ]; then echo INFERENCE_ROUTE_UNAVAILABLE; fi'
)
run_command(
    f"openshell sandbox exec -n {course_sandbox_name} --timeout 30 -- sh -lc {shlex.quote(inference_probe)}",
    timeout=40,
)

---

## 9. Preview Host Aliases

Current NemoClaw exposes host-alias lifecycle commands for cases where a sandbox needs a stable LAN-only hostname for an internal service. Use `--dry-run` first so the patch is visible before anything changes.


In [ ]:
run_command(f"nemoclaw {course_sandbox_name} hosts-list --help", timeout=30)
run_command(f"nemoclaw {course_sandbox_name} hosts-add --help", timeout=30)

---

## Checkpoint

1. **Check:** the sandbox list shows the course sandbox as Ready.
   - **Expected:** a sandbox named `agent` or another detected course name is Ready.
   - **If not working:** rerun `./setup.sh` from the repository root before continuing.

2. **Check:** OpenClaw is found inside the sandbox.
   - **Expected:** `command -v openclaw` prints `/usr/local/bin/openclaw` and the gateway health command either returns OK or an actionable pairing message.
   - **If not working:** inspect `nemoclaw status` and the sandbox logs.

3. **Check:** `/sandbox/.openclaw/openclaw.json` exists.
   - **Expected:** the current state root is `/sandbox/.openclaw`, not the legacy data path.

4. **Check:** workspace inspection prints files or a clear empty-workspace message.
   - **Expected:** files such as identity, soul, memory, or workspace-scoped skill files may appear depending on the target.
   - **If not working:** confirm the sandbox name and state root from the earlier sections.

5. **Check:** skills list returns a readable list rather than a shell error.
   - **Expected:** the count may vary as OpenClaw releases change, but the command should run.

6. **Check:** `openshell policy get` prints the active policy.
   - **Expected:** network, filesystem, process, and inference-related policy domains may appear depending on the target.

7. **Check:** the inference route does not reveal the API key.
   - **Expected:** model JSON or an actionable policy/provider error. A raw secret in output is a blocker.

---

## ✅ What You Accomplished

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Verified the default NemoClaw blueprint as three cooperating layers: NemoClaw lifecycle, OpenShell runtime, and OpenClaw harness</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Confirmed OpenClaw runs inside the sandbox with its state under <code>/sandbox/.openclaw</code></p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Inspected the OpenClaw workspace, skills, and session surfaces without mutating the environment</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Read the live OpenShell policy on the managed sandbox</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Probed the inference route and confirmed it does not reveal the API key</p>

## Next Steps

Continue to **[NASA APOD: Your First API-backed Skill](../04-integrations/01_nasa_apod.ipynb)** -- you'll review a skill, add the network policy it needs, and deploy it to an agent in this sandbox.

---

<small>

**Sources:**
[NemoClaw commands](https://docs.nvidia.com/nemoclaw/latest/user-guide/openclaw/reference/commands) ·
[NemoClaw security best practices](https://docs.nvidia.com/nemoclaw/latest/user-guide/openclaw/security/best-practices) ·
[NemoClaw blueprint v0.0.79](https://github.com/NVIDIA/NemoClaw/blob/v0.0.79/nemoclaw-blueprint/blueprint.yaml) ·
[NemoClaw OpenClaw manifest](https://github.com/NVIDIA/NemoClaw/blob/v0.0.79/agents/openclaw/manifest.yaml) ·
[OpenShell sandboxes](https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes) ·
[OpenShell policy schema](https://docs.nvidia.com/openshell/reference/policy-schema) ·
[OpenClaw agent workspace](https://docs.openclaw.ai/concepts/agent-workspace) ·
[OpenClaw gateway docs](https://docs.openclaw.ai/gateway/index)

</small>